In [1]:
import numpy as np
import pickle
import os
import time

# ===========================================================
# =============== MoSS BINÁRIO (BETA DISTRIBUTION) ==========
# ===========================================================

def moss_binary_beta(n_samples: int, p_pos: float, merge: float, eps: float = 1e-3):
    """
    Versão binária do MoSS usando distribuição Beta.
    p_pos: prevalência da classe positiva (0.0 a 1.0)
    """
    merge = np.clip(merge, 0.0, 1.0)
    # Scale controla a "certeza" do classificador. 
    # Quanto menor o merge, mais as amostras se agrupam nos extremos (0 e 1).
    scale = 50 * (1 - merge) + 5
    
    n_pos = int(np.floor(n_samples * p_pos))
    n_neg = n_samples - n_pos
    
    scores = np.zeros((n_samples, 2)) # Formato [prob_neg, prob_pos]

    # Amostras da Classe Negativa (tendem a score 0)
    if n_neg > 0:
        # Alpha < Beta -> cauda para a esquerda (próximo de 0)
        a_neg = eps + scale * merge
        b_neg = eps + scale * (1 - merge + merge) # Ajuste de dispersão
        s_neg = np.random.beta(a_neg, b_neg + scale * (1-merge), size=n_neg)
        scores[:n_neg, 1] = s_neg
        scores[:n_neg, 0] = 1 - s_neg

    # Amostras da Classe Positiva (tendem a score 1)
    if n_pos > 0:
        # Alpha > Beta -> cauda para a direita (próximo de 1)
        a_pos = eps + scale * (1 - merge + merge)
        b_pos = eps + scale * merge
        s_pos = np.random.beta(a_pos + scale * (1-merge), b_pos, size=n_pos)
        scores[n_neg:, 1] = s_pos
        scores[n_neg:, 0] = 1 - s_pos

    np.random.shuffle(scores)
    return scores

# ===========================================================
# ========== GERADOR DE DISTRIBUIÇÕES MOSS BINÁRIO ==========
# ===========================================================

def gerar_distribuicoes_moss_binario(n_samples, n_prevalences, n_merges, n_curves, save_path):
    # No binário, a prevalência é apenas um float entre 0 e 1
    prevalences = np.linspace(0.0, 1.0, n_prevalences)
    merges = np.linspace(0.0, 1.0, n_merges)
    synthetic_distributions = {}
    total = len(prevalences) * len(merges)
    count = 0

    print(f"\n🚀 Gerando MoSS BINÁRIO -> {save_path}")
    start_global = time.perf_counter()

    for p in prevalences:
        # Mantemos o formato de tupla para compatibilidade de chave (p_neg, p_pos)
        alpha_key = (round(1-p, 4), round(p, 4))
        for merge in merges:
            curves = [moss_binary_beta(n_samples, p, merge) for _ in range(n_curves)]
            synthetic_distributions[(alpha_key, round(merge, 4))] = curves
            count += 1
            if count % 50 == 0 or count == total:
                print(f"   Progresso: [{count}/{total}] blocos concluídos...")

    with open(save_path, "wb") as f:
        pickle.dump(synthetic_distributions, f, protocol=pickle.HIGHEST_PROTOCOL)

    total_runtime = time.perf_counter() - start_global
    print(f"✔ Finalizado MoSS Binário em {total_runtime/60:.2f} min\n")

# ===========================================================
# ======================= EXECUÇÃO ==========================
# ===========================================================

if __name__ == "__main__":
    output_dir = "moss_outputs"
    os.makedirs(output_dir, exist_ok=True)

    # Suas configurações solicitadas
    config = dict(
        n_samples=500,
        n_prevalences=15,
        n_merges=15,
        n_curves=20
    )

    file_path = os.path.join(output_dir, "moss_binario_lite.pkl")
    
    gerar_distribuicoes_moss_binario(
        save_path=file_path,
        **config
    )

    print(f"🏁 Arquivo binário gerado: {file_path}")


🚀 Gerando MoSS BINÁRIO -> moss_outputs/moss_binario_lite.pkl
   Progresso: [50/225] blocos concluídos...
   Progresso: [100/225] blocos concluídos...
   Progresso: [150/225] blocos concluídos...
   Progresso: [200/225] blocos concluídos...
   Progresso: [225/225] blocos concluídos...
✔ Finalizado MoSS Binário em 0.02 min

🏁 Arquivo binário gerado: moss_outputs/moss_binario_lite.pkl
